# Peer Effect Standardized Regression Analysis

## Objective
To determine the **relative influence** of Peer Depression vs. Personal History on future depression.

## Methodology
We use **Standardized Linear Regression** (Beta Coefficients).
All variables are converted to Z-scores (Mean=0, Std=1) before analysis.
- **Beta**: Represents the change in outcome (in standard deviations) for a 1-SD change in the predictor.
- **Ratio**: Beta(Own) / Beta(Peer) tells us how much stronger one factor is compared to the other.

---

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
from scipy.stats import zscore

# --- configuration ---
# Paths 
W2_PEER_STATS_PATH = r"../../relationship_reallike/w2_peer_mental_health_stats.csv"
W3_DATA_PATH = r"../../../../Data/2025data/TIGPS_W3_studentdata_ver4_cleaned_cols_removed_missing_common_only.csv"

# Fix absolute paths if running locally/interactively
if not os.path.exists(W2_PEER_STATS_PATH):
    W2_PEER_STATS_PATH = r"C:/Users/user/Desktop/TIGPS_Plan_data/20251229_new_progress/Code/EDA/relationship_reallike/w2_peer_mental_health_stats.csv"
    W3_DATA_PATH = r"C:/Users/user/Desktop/TIGPS_Plan_data/20251229_new_progress/Data/2025data/TIGPS_W3_studentdata_ver4_cleaned_cols_removed_missing_common_only.csv"

print(f"Loading Peer Stats from: {W2_PEER_STATS_PATH}")
print(f"Loading W3 Data from: {W3_DATA_PATH}")

Loading Peer Stats from: ../../relationship_reallike/w2_peer_mental_health_stats.csv
Loading W3 Data from: ../../../../Data/2025data/TIGPS_W3_studentdata_ver4_cleaned_cols_removed_missing_common_only.csv


In [2]:
# 1. Load Data
peer_df = pd.read_csv(W2_PEER_STATS_PATH)
try:
    w3_df = pd.read_csv(W3_DATA_PATH, on_bad_lines='skip', engine='python')
except Exception as e:
    print(f"Error loading W3 data: {e}")

# 2. Calculate W3 Mental Health Score
mh_cols_w3 = [f"54-{i}" for i in range(1, 15)]
w3_df['w3_mh_score'] = w3_df[mh_cols_w3].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)
w3_df = w3_df.dropna(subset=['w3_mh_score', 'student_id'])

# 3. Merge Datasets
merged = pd.merge(peer_df, w3_df[['student_id', 'w3_mh_score']], on='student_id', how='inner')
print(f"Merged Data: {merged.shape[0]} students")

Merged Data: 6733 students


In [3]:
# 4. Prepare Regression Data
X = merged[['w2_own_mh_score', 'w2_peer_avg_mh_score']]
X = sm.add_constant(X) 
y = merged['w3_mh_score']

data_reg = pd.concat([X, y], axis=1).dropna()
print(f"Final Samples: {len(data_reg)}")

Final Samples: 6593


In [4]:
# 5. Standardization (Z-score)
print("Standardizing variables to Z-scores...")
data_std = data_reg.copy()
cols_to_std = ['w2_own_mh_score', 'w2_peer_avg_mh_score', 'w3_mh_score']

for col in cols_to_std:
    mean_val = data_std[col].mean()
    std_val = data_std[col].std()
    data_std[col] = (data_std[col] - mean_val) / std_val
    print(f"  {col}: Mean={mean_val:.2f}, Std={std_val:.2f}")

X_std = data_std[['const', 'w2_own_mh_score', 'w2_peer_avg_mh_score']]
y_std = data_std['w3_mh_score']

Standardizing variables to Z-scores...
  w2_own_mh_score: Mean=21.38, Std=10.29
  w2_peer_avg_mh_score: Mean=21.36, Std=6.65
  w3_mh_score: Mean=22.39, Std=11.56


In [5]:
# 6. Run Standardized OLS Regression
model_std = sm.OLS(y_std, X_std).fit()

print(model_std.summary())

                            OLS Regression Results                            
Dep. Variable:            w3_mh_score   R-squared:                       0.196
Model:                            OLS   Adj. R-squared:                  0.195
Method:                 Least Squares   F-statistic:                     801.7
Date:                Tue, 13 Jan 2026   Prob (F-statistic):          2.22e-312
Time:                        16:26:35   Log-Likelihood:                -8636.6
No. Observations:                6593   AIC:                         1.728e+04
Df Residuals:                    6590   BIC:                         1.730e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                -1.246e-15 

1. 模型的整體表現 (Model Fit)
Dep. Variable (依變項): w3_mh_score (預測目標：W3 憂鬱分數)
R-squared: 0.196 (約 0.20)
解釋：這個模型解釋了 W3 學生憂鬱分數變異的 19.6%。
結論：這個數字跟我們之前純用 ML 跑出來的 (0.24) 差不多，甚至稍微低一點。說明大部分的未來變化是無法單純用這兩個變數解釋的。
2. 關鍵變數的影響力 (Coefficients)
這是最精彩的部分。因為數據已標準化，coef 代表 Beta 係數 (影響力大小)：

w2_own_mh_score (自己去年的分數)
Coef: 0.4365
P>|t|: 0.000 (超級顯著)
解釋：自己去年的狀態是今年最強的預測因子。「慣性」很強。


w2_peer_avg_mh_score (朋友去年的分數)
Coef: 0.0266 (重點！)
P>|t|: 0.018 (小於 0.05，統計上顯著)
解釋：雖然統計上「有影響」，但影響力非常小 (只有 0.02)。


對比：記得我們之前跑 W2->W2 (共時) 的時候，朋友的影響力係數高達 0.19 嗎？
現在進行式 (W2->W2)：朋友影響力 = 0.19 (很強)
預測未來式 (W2->W3)：朋友影響力 = 0.02 (剩下一點點)

In [6]:
# 7. Result Interpretation
beta_own = model_std.params['w2_own_mh_score']
beta_peer = model_std.params['w2_peer_avg_mh_score']
pval_peer = model_std.pvalues['w2_peer_avg_mh_score']

print(f"\n--- Key Findings (Standardized) ---")
print(f"Beta (Own History): {beta_own:.4f}")
print(f"Beta (Peer Influence): {beta_peer:.4f}")
print(f"P-value (Peer): {pval_peer:.4e}")

if beta_peer > 0:
    ratio = beta_own / beta_peer
    print(f"\n>> Relative Strength: Your own past history is {ratio:.1f}x stronger than peer influence.")
    print(f">> However, peer influence is still statistically significant (p < 0.05).")


--- Key Findings (Standardized) ---
Beta (Own History): 0.4365
Beta (Peer Influence): 0.0266
P-value (Peer): 1.8238e-02

>> Relative Strength: Your own past history is 16.4x stronger than peer influence.
>> However, peer influence is still statistically significant (p < 0.05).
